In [ ]:
import pandas as pd
years = [2016, 2017, 2018, 2020, 2021, 2022, 2023]
bigset = []
for year in years:
    df = pd.read_csv(f"BRFSS{year}_cleaned.csv")
    bigset.append(df)
full_df = pd.concat(bigset, ignore_index = True)
#full_df.to_csv("Full.csv", index = False)


In [ ]:
full_df = pd.read_csv("Full.csv")
full_df.head()
#full_df['survey_date'] = pd.to_datetime(full_df['DATE'], format='%m%d%Y', errors='coerce')


In [20]:
full_df['_STATE'] = full_df['_STATE'].astype(str).str.zfill(2)
full_df['survey_date'] = pd.to_datetime(full_df['survey_date'])
full_df['survey_year'] = full_df['survey_date'].dt.year
full_df['survey_month'] = full_df['survey_date'].dt.month
full_df.head()

,_STATE,DATE,_AGE80,SEXVAR,_RACE,MENTHLTH,ECIGNOW2,USENOW3,_RFSMOK3,_RFBING6,MARIJAN1,LLCPWT,survey_date,survey_year,survey_month
0,01,2032016,42,1,1.0,88.0,2.0,3.0,2,1,NaN,1141.248720,2016-02-03,2016.0,2.0
1,01,2032016,70,2,1.0,10.0,4.0,3.0,2,1,NaN,531.192398,2016-02-03,2016.0,2.0
2,01,3102016,64,1,1.0,88.0,4.0,3.0,2,1,NaN,413.232363,2016-03-10,2016.0,3.0
3,01,3022016,51,1,1.0,30.0,4.0,3.0,2,1,NaN,243.946280,2016-03-02,2016.0,3.0
4,01,1102016,35,1,1.0,88.0,4.0,3.0,2,1,NaN,1339.733850,2016-01-10,2016.0,1.0


In [21]:
unemp = pd.read_csv("Unemployment.csv")
unemp['date'] = pd.to_datetime(unemp['Attribute'], format='%b %Y')

# Extract month/year
unemp['year'] = unemp['date'].dt.year
unemp['month'] = unemp['date'].dt.month
unemp['STATE'] = unemp['STATE'].astype(str).str.zfill(2)
unemp.head()

,STATE,Attribute,Unemployment Rate,date,year,month
0,01,Jan\n2016,5.9,2016-01-01,2016,1
1,01,Feb\n2016,5.9,2016-02-01,2016,2
2,01,Mar\n2016,5.9,2016-03-01,2016,3
3,01,Apr\n2016,5.8,2016-04-01,2016,4
4,01,May\n2016,5.8,2016-05-01,2016,5


In [24]:
merged = full_df.merge(
    unemp,
    left_on=['_STATE', 'survey_year', 'survey_month'],
    right_on=['STATE', 'year', 'month'],
    how='left'
)
merged = merged.drop(columns = ['DATE', 'STATE', 'Attribute', 'date', 'year', 'month'])
merged.head()


,_STATE,_AGE80,SEXVAR,_RACE,MENTHLTH,ECIGNOW2,USENOW3,_RFSMOK3,_RFBING6,MARIJAN1,LLCPWT,survey_date,survey_year,survey_month,Unemployment Rate
0,01,42,1,1.0,88.0,2.0,3.0,2,1,NaN,1141.248720,2016-02-03,2016.0,2.0,5.9
1,01,70,2,1.0,10.0,4.0,3.0,2,1,NaN,531.192398,2016-02-03,2016.0,2.0,5.9
2,01,64,1,1.0,88.0,4.0,3.0,2,1,NaN,413.232363,2016-03-10,2016.0,3.0,5.9
3,01,51,1,1.0,30.0,4.0,3.0,2,1,NaN,243.946280,2016-03-02,2016.0,3.0,5.9
4,01,35,1,1.0,88.0,4.0,3.0,2,1,NaN,1339.733850,2016-01-10,2016.0,1.0,5.9


In [25]:
covid = pd.read_csv("covid_deaths_usafacts.csv")
agg_covid = covid.groupby("StateFIPS").sum(numeric_only=True).reset_index()
agg_covid.head()
agg_covid.to_csv("aggregatedCovid.csv", index=False)

In [26]:
final_covid = pd.read_csv("CovidDeath_state_month.csv")
final_covid['year'] = final_covid['YearMonth'].str[:4].astype(int)
final_covid['month'] = final_covid['YearMonth'].str[5:7].astype(int)
final_covid.head()


,STATE,YearMonth,Death,year,month
0,1,2020-01,0,2020,1
1,1,2020-02,0,2020,2
2,1,2020-03,13,2020,3
3,1,2020-04,256,2020,4
4,1,2020-05,360,2020,5


In [28]:
merged['state'] = merged['_STATE'].astype(str).str.zfill(2)
final_covid['state'] = final_covid['STATE'].astype(str).str.zfill(2)
merged = merged.dropna(subset=['survey_year'])
merged['survey_year'] = merged['survey_year'].astype(int)
merged['survey_month'] = merged['survey_month'].astype(int)
final_covid['year'] = final_covid['year'].astype(int)
final_covid['month'] = final_covid['month'].astype(int)

main_df = merged.merge(
    final_covid[['state', 'year', 'month', 'Death']],
    left_on=['state', 'survey_year', 'survey_month'],
    right_on=['state', 'year', 'month'],
    how='left'
)
main_df = main_df.drop(columns = ["year", "month", "state"])
main_df['Death'] = main_df['Death'].fillna(0)
main_df.head()


,_STATE,_AGE80,SEXVAR,_RACE,MENTHLTH,ECIGNOW2,USENOW3,_RFSMOK3,_RFBING6,MARIJAN1,LLCPWT,survey_date,survey_year,survey_month,Unemployment Rate,Death
0,01,42,1,1.0,88.0,2.0,3.0,2,1,NaN,1141.248720,2016-02-03,2016,2,5.9,0.0
1,01,70,2,1.0,10.0,4.0,3.0,2,1,NaN,531.192398,2016-02-03,2016,2,5.9,0.0
2,01,64,1,1.0,88.0,4.0,3.0,2,1,NaN,413.232363,2016-03-10,2016,3,5.9,0.0
3,01,51,1,1.0,30.0,4.0,3.0,2,1,NaN,243.946280,2016-03-02,2016,3,5.9,0.0
4,01,35,1,1.0,88.0,4.0,3.0,2,1,NaN,1339.733850,2016-01-10,2016,1,5.9,0.0


In [29]:
#flavro bans
flavor_ban_dates = {
    '06': '2022-01-01',  # California
    '25': '2019-09-24',  # 2
    '36': '2020-05-18',  # New York
    '34': '2020-01-01',  # New Jersey
    '44': '2020-03-01',  # Rhode Island
    '24': '2021-01-01',  # Maryland
    '23': '2021-01-01',  # Maine
    '49': '2020-01-01',  # Utah
}
main_df['survey_date'] = pd.to_datetime(main_df['survey_date'])
main_df['flavor_ban'] = 0 
for fips, start_date in flavor_ban_dates.items():
    date = pd.to_datetime(start_date)
    mask = (main_df['_STATE'] == fips) & (main_df['survey_date'] >= date)
    main_df.loc[mask, 'flavor_ban'] = 1

In [30]:
main_df['flavor_ban'].value_counts()

flavor_ban
0    1210421
1     269111
Name: count, dtype: int64

In [31]:
#T21
t21_dates = {
    '06': '2016-06-09',
    '15': '2016-01-01',
    '34': '2017-11-01',
    '41': '2018-01-01',
    '23': '2018-07-01',
    '25': '2018-12-31',
    '17': '2019-07-01',
    '51': '2019-07-01',
    '05': '2019-09-01',
    '48': '2019-09-01',
    '10': '2019-09-10',
    '09': '2019-10-01',
    '24': '2019-10-01',
    '36': '2019-11-13',
    '53': '2020-01-01'
}
default_t21_date = pd.to_datetime('2019-12-20')
main_df['t21'] = 0  # initialize
for fips, start_date in t21_dates.items():
    date = pd.to_datetime(start_date)
    mask = (main_df['_STATE'] == fips) & (main_df['survey_date'] >= date)
    main_df.loc[mask, 't21'] = 1
mask_fed = (~main_df['_STATE'].isin(t21_dates.keys())) & (main_df['survey_date'] >= default_t21_date)
main_df.loc[mask_fed, 't21'] = 1
main_df['t21'].value_counts()


t21
1    1300217
0     179315
Name: count, dtype: int64

In [32]:
cigarette_tax = pd.read_csv("combustible_cigarette_tax.csv")
cigarette_tax['state'] = cigarette_tax['state'].astype(str).str.zfill(2)
cigarette_tax['year'] = cigarette_tax['date'].str[:4].astype(int)

main_df = main_df.merge(
    cigarette_tax[['state', 'year', 'tax']],
    left_on=['_STATE', 'survey_year'],
    right_on=['state', 'year'],
    how='left'
)
main_df = main_df.drop(columns = ["state", "year"])



In [33]:
main_df.head()

,_STATE,_AGE80,SEXVAR,_RACE,MENTHLTH,ECIGNOW2,USENOW3,_RFSMOK3,_RFBING6,MARIJAN1,LLCPWT,survey_date,survey_year,survey_month,Unemployment Rate,Death,flavor_ban,t21,tax
0,01,42,1,1.0,88.0,2.0,3.0,2,1,NaN,1141.248720,2016-02-03,2016,2,5.9,0.0,0,0,0.675
1,01,70,2,1.0,10.0,4.0,3.0,2,1,NaN,531.192398,2016-02-03,2016,2,5.9,0.0,0,0,0.675
2,01,64,1,1.0,88.0,4.0,3.0,2,1,NaN,413.232363,2016-03-10,2016,3,5.9,0.0,0,0,0.675
3,01,51,1,1.0,30.0,4.0,3.0,2,1,NaN,243.946280,2016-03-02,2016,3,5.9,0.0,0,0,0.675
4,01,35,1,1.0,88.0,4.0,3.0,2,1,NaN,1339.733850,2016-01-10,2016,1,5.9,0.0,0,0,0.675


In [34]:
print(main_df['survey_date'].max())

2024-12-09 00:00:00


In [35]:
ends_tax_dates = {
    "01": "2020-01-01",  # Alabama
    "32": "2020-01-01",  # Nevada
    "02": "2020-01-01",  # Alaska
    "33": "2020-01-01",  # New Hampshire
}

main_df['ecig_tax'] = 0
main_df['survey_date'] = pd.to_datetime(main_df['survey_date'])

for fips, date in ends_tax_dates.items():
    main_df.loc[
        (main_df['_STATE'] == fips) & 
        (main_df['survey_date'] >= pd.to_datetime(date)),
        'ecig_tax'
    ] = 1

In [37]:
import numpy as np
main_df['_RACE'] = main_df['_RACE'].replace(9, np.nan)

In [38]:

main_df['MENTHLTH'] = main_df['MENTHLTH'].replace(88, 0)
main_df['MENTHLTH'] = main_df['MENTHLTH'].replace(77, np.nan)
main_df['MENTHLTH'] = main_df['MENTHLTH'].replace(99, np.nan)

In [39]:
main_df['ECIGNOW2'] = main_df['ECIGNOW2'].replace(7, np.nan)
main_df['ECIGNOW2'] = main_df['ECIGNOW2'].replace(9, np.nan)

In [40]:
main_df['USENOW3'].unique()
main_df['USENOW3'] = main_df['USENOW3'].replace(7, np.nan)
main_df['USENOW3'] = main_df['USENOW3'].replace(9, np.nan)

In [41]:
main_df['_RFSMOK3'].unique()
main_df['_RFSMOK3'] = main_df['_RFSMOK3'].replace(9, np.nan)

In [42]:
main_df['_RFBING6'].unique()
main_df['_RFBING6'] = main_df['_RFBING6'].replace(9, np.nan)


In [43]:
main_df['MARIJAN1'] = main_df['MARIJAN1'].replace(88, 0)
main_df['MARIJAN1'] = main_df['MARIJAN1'].replace(77, np.nan)
main_df['MARIJAN1'] = main_df['MARIJAN1'].replace(99, np.nan)

In [44]:
main_df_clean = main_df.dropna(subset=['ECIGNOW2'])
main_df_clean.to_csv("LEZGOOO.csv")